In [ ]:
!pip install scikit-learn==1.6.1 scipy==1.16.3 -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

BASE_FEAT_BIN   = '/content/drive/MyDrive/riset_teks/features-biner'
BASE_RESULT_BIN = '/content/drive/MyDrive/riset_teks/results-biner'

datasets_biner = ['dataset1', 'dataset3']
methods        = ['tfidf', 'word2vec', 'bert']

for ds in datasets_biner:
    for method in methods:
        for level in range(5):
            os.makedirs(f"{BASE_FEAT_BIN}/{ds}/{method}/level{level}", exist_ok=True)
    for method in methods:
        for level in range(5):
            os.makedirs(f"{BASE_RESULT_BIN}/{ds}/{method}/level{level}", exist_ok=True)

os.makedirs(BASE_RESULT_BIN, exist_ok=True)
print("done!")

In [ ]:
import numpy as np

def remap_label_dataset1(y):
    """
    Positive (1) : Love=0, Happy=1
    Negative (0) : Anger=2, Fear=3, Sadness=4
    """
    y_bin = np.where(np.isin(y, [0, 1]), 1, 0)
    return y_bin

def remap_label_dataset3(y):
    """
    Cancer     (1) : Neoplasms=1
    Non-Cancer (0) : Digestive=2, Nervous=3, Cardiovascular=4, General=5
    """
    y_bin = np.where(y == 1, 1, 0)
    return y_bin

REMAP_FN = {
    'dataset1': remap_label_dataset1,
    'dataset3': remap_label_dataset3,
}

print("done!")

In [ ]:
import pandas as pd
import scipy.sparse as sp
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

BASE_PREP       = '/content/drive/MyDrive/riset_teks/preprocessed'
BASE_FEAT       = '/content/drive/MyDrive/riset_teks/features'
BASE_FEAT_BIN   = '/content/drive/MyDrive/riset_teks/features-biner'
BASE_RESULT_BIN = '/content/drive/MyDrive/riset_teks/results-biner'

methods  = ['tfidf', 'word2vec', 'bert']
summary_biner_rows = []

for dataset in ['dataset1', 'dataset3']:
    remap_fn = REMAP_FN[dataset]
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset} — MODE BINER")
    print(f"{'='*60}")

    for level in range(5):
        print(f"\n  ── Level {level} ──")

        for fold in range(1, 6):
            print(f"    Fold {fold}...")

            # Load y from CSV, then remap
            train_df = pd.read_csv(
                f"{BASE_PREP}/{dataset}/level{level}/fold{fold}_train.csv"
            )
            test_df  = pd.read_csv(
                f"{BASE_PREP}/{dataset}/level{level}/fold{fold}_test.csv"
            )
            y_train = remap_fn(train_df['class'].values)
            y_test  = remap_fn(test_df['class'].values)

            row = {'dataset': dataset, 'level': level, 'fold': fold}

            for method in methods:
                feat_path = f"{BASE_FEAT}/{dataset}/{method}/level{level}"

                # Load X
                if method == 'tfidf':
                    X_train = sp.load_npz(f"{feat_path}/fold{fold}_train.npz")
                    X_test  = sp.load_npz(f"{feat_path}/fold{fold}_test.npz")
                else:
                    X_train = np.load(f"{feat_path}/fold{fold}_train_X.npy")
                    X_test  = np.load(f"{feat_path}/fold{fold}_test_X.npy")

                # Save to features-biner
                out_path = f"{BASE_FEAT_BIN}/{dataset}/{method}/level{level}"
                if sp.issparse(X_train):
                    sp.save_npz(f"{out_path}/fold{fold}_train.npz", X_train)
                    sp.save_npz(f"{out_path}/fold{fold}_test.npz",  X_test)
                else:
                    np.save(f"{out_path}/fold{fold}_train_X.npy", X_train)
                    np.save(f"{out_path}/fold{fold}_test_X.npy",  X_test)
                np.save(f"{out_path}/fold{fold}_train_y.npy", y_train)
                np.save(f"{out_path}/fold{fold}_test_y.npy",  y_test)

                # SVM
                clf = LinearSVC(
                    C=1.0,
                    class_weight='balanced',
                    max_iter=5000,
                    random_state=42,
                    dual='auto'
                )
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)

                metrics = {
                    'accuracy' : accuracy_score(y_test, y_pred),
                    'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
                    'recall'   : recall_score(y_test, y_pred, average='macro', zero_division=0),
                    'f1'       : f1_score(y_test, y_pred, average='macro', zero_division=0),
                }

                # save report
                report     = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
                report_df  = pd.DataFrame(report).transpose()
                res_path   = f"{BASE_RESULT_BIN}/{dataset}/{method}/level{level}"
                report_df.to_csv(f"{res_path}/fold{fold}_report.csv")

                # Collect to row summary
                pfx = method if method != 'word2vec' else 'w2v'
                row[f"{pfx}_acc"]  = metrics['accuracy']
                row[f"{pfx}_prec"] = metrics['precision']
                row[f"{pfx}_rec"]  = metrics['recall']
                row[f"{pfx}_f1"]   = metrics['f1']

            summary_biner_rows.append(row)
        print(f"  Level {level} completed!")

    print(f"\n  binery {dataset} completed!")

print("\n all binery datasets are completed!")

In [ ]:
summary_biner_df = pd.DataFrame(summary_biner_rows)
summary_biner_df.to_csv(f"{BASE_RESULT_BIN}/summary_biner_all_folds.csv", index=False)

methods_pfx  = ['tfidf', 'w2v', 'bert']
method_label = ['TF-IDF', 'Word2Vec', 'BERT']
metrics      = ['acc', 'prec', 'rec', 'f1']
metric_label = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

rows_agg = []
for (dataset, level), grp in summary_biner_df.groupby(['dataset', 'level']):
    row = {'dataset': dataset, 'level': level}
    for pfx in methods_pfx:
        for m in metrics:
            col = f"{pfx}_{m}"
            row[f"{col}_mean"] = round(grp[col].mean(), 4)
            row[f"{col}_std"]  = round(grp[col].std(),  4)
    rows_agg.append(row)

result_biner_df = pd.DataFrame(rows_agg)
result_biner_df.to_csv(f"{BASE_RESULT_BIN}/summary_biner_mean_std.csv", index=False)

for dataset in ['dataset1', 'dataset3']:
    print(f"\n{'='*70}")
    print(f"  DATASET: {dataset} (BINER)")
    print(f"{'='*70}")
    sub = result_biner_df[result_biner_df['dataset'] == dataset].set_index('level')

    for pfx, mlabel in zip(methods_pfx, method_label):
        print(f"\n  [{mlabel}]")
        print(f"  {'Level':<8}", end="")
        for ml in metric_label:
            print(f"  {ml:<22}", end="")
        print()
        print(f"  {'-'*100}")
        for lvl in range(5):
            print(f"  {lvl:<8}", end="")
            for m in metrics:
                mean = sub.loc[lvl, f"{pfx}_{m}_mean"]
                std  = sub.loc[lvl, f"{pfx}_{m}_std"]
                print(f"  {mean:.4f} ± {std:.4f}      ", end="")
            print()

print(f"\n Summary: {BASE_RESULT_BIN}")